<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_1_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.1-A — FIXED-R ACCURACY AND POPULATION-SIZE GENERALIZATION
# FINAL / PUBLICATION-LEVEL / RESUMABLE / CPU-ONLY
#
# R = 10,000 exact-teacher configurations
# 5 independent neural-training replications
#
# Training population sizes:
#   N = 40,60,...,400
#
# Held-out interpolation population sizes:
#   N = 50,70,...,390
#
# Main scientific questions:
#   1. How accurately does one population-size-coherent emulator reproduce
#      the exact truncated-with-overflow infection-count distribution?
#   2. Is tail-risk accuracy preserved?
#   3. Is there an interpolation penalty at population sizes never used
#      for teacher construction?
#   4. Are conclusions robust across epidemic regimes and i0=1 cases?
#
# Main outputs:
#   main_table_5_1A.tex
#   figure_5_1A_main.pdf
#
# Additional:
#   full_numeric_summary.csv
#   raw_test_errors.csv
#   accuracy_by_N.csv
#   interpolation_contrast.csv
#   robustness_by_stratum.csv
#   training_diagnostics.csv
#   tau_resolution.csv
#
# CPU ONLY | SPARSE LU | NO MATRIX INVERSE
# =====================================================================================


# =====================================================================================
# 0. DRIVE + ENVIRONMENT
# =====================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
os.environ["OMP_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
os.environ["NUMEXPR_NUM_THREADS"]="1"

import json, hashlib, math, pickle, random, time
from dataclasses import dataclass, asdict
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import matplotlib.pyplot as plt


# =====================================================================================
# 1. CONFIG
# =====================================================================================

@dataclass
class Config:
    seed:int = 20260820

    beta:Tuple[float,float] = (.30,1.50)
    gamma:Tuple[float,float] = (.20,1.00)
    omega:Tuple[float,float] = (.02,.50)
    frac:Tuple[float,float] = (.02,.20)

    train_N:Tuple[int,...] = tuple(range(40,401,20))
    interp_N:Tuple[int,...] = tuple(range(50,400,20))

    n_train:int = 10000

    width:int = 128
    depth:int = 3
    batch:int = 64
    epochs:int = 500

    lr:float = 1e-3
    wd:float = 1e-6

    lambda_rho:float = 1.
    lambda_tau:float = .02

    patience:int = 20
    delta:float = 1e-6
    clip:float = 5.

    prob_tol:float = 1e-10
    var_tol:float = 1e-8
    refine:int = 3


cfg=Config()

# Equal coverage within each population-size test grid.
VAL_PER_N=25
TEST_PER_N=25

N_VAL=len(cfg.train_N)*VAL_PER_N             # 475
N_TEST_SEEN=len(cfg.train_N)*TEST_PER_N     # 475
N_TEST_INTERP=len(cfg.interp_N)*TEST_PER_N  # 450

N_REP=5
BOOT_B=3000

EXACT_CHUNK=50
CHECKPOINT_EVERY=5

N_SCALE=max(cfg.train_N)

assert set(cfg.train_N).isdisjoint(cfg.interp_N)


# =====================================================================================
# 2. RESUME / START-NEW PROTECTION
#
# Increment CODE_VERSION whenever the scientific implementation changes:
# exact solver, loss, architecture, target definition, or design logic.
# Plotting-only changes do not require a new version.
# =====================================================================================

CODE_VERSION="5.1A_JASA_v1"

SCIENTIFIC_CONFIG={
    "code_version":CODE_VERSION,
    "config":asdict(cfg),
    "VAL_PER_N":VAL_PER_N,
    "TEST_PER_N":TEST_PER_N,
    "N_REP":N_REP,
    "protocol":"fixed exact training set; independent optimization replications"
}

def make_signature(x):
    txt=json.dumps(x,sort_keys=True,default=str)
    return hashlib.sha256(txt.encode()).hexdigest()[:16]

SIG=make_signature(SCIENTIFIC_CONFIG)

BASE=Path(
    "/content/drive/MyDrive/StatisticalLearning/"
    "Experiment_5_1A_JASA"
)
BASE.mkdir(parents=True,exist_ok=True)

print("\n"+"="*82)
print("1 = RESUME latest run")
print("2 = START NEW run from zero")
print("="*82)

mode=input("Choose 1 or 2: ").strip()

if mode not in ("1","2"):
    raise RuntimeError("Choose 1 or 2.")

latest=BASE/"latest_run.txt"

if mode=="2":
    stamp=datetime.now().strftime("%Y%m%d_%H%M%S")
    ROOT=BASE/f"run_{stamp}"
    ROOT.mkdir()

    with open(ROOT/"manifest.pkl","wb") as f:
        pickle.dump(
            {"signature":SIG,"scientific_config":SCIENTIFIC_CONFIG},
            f,pickle.HIGHEST_PROTOCOL
        )

    latest.write_text(ROOT.name)

else:
    if not latest.exists():
        raise RuntimeError("No previous run. Choose START NEW.")

    ROOT=BASE/latest.read_text().strip()
    manifest_file=ROOT/"manifest.pkl"

    if not manifest_file.exists():
        raise RuntimeError("Missing manifest; resume is unsafe.")

    with open(manifest_file,"rb") as f:
        old=pickle.load(f)

    if old["signature"]!=SIG:
        raise RuntimeError(
            "Scientific specification changed. "
            "RESUME aborted; choose START NEW."
        )


CACHE=ROOT/"cache"
EXACT=CACHE/"exact"
MODELS=CACHE/"models"
CKPT=CACHE/"checkpoints"
OUT=ROOT/"results"

for d in (CACHE,EXACT,MODELS,CKPT,OUT):
    d.mkdir(parents=True,exist_ok=True)

print("ROOT:",ROOT)
print("Signature:",SIG)


# =====================================================================================
# 3. UTILITIES / CPU
# =====================================================================================

def atomic_pickle(x,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+".tmp")
    with open(tmp,"wb") as f:
        pickle.dump(x,f,pickle.HIGHEST_PROTOCOL)
    os.replace(tmp,path)

def atomic_torch(x,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+".tmp")
    torch.save(x,tmp)
    os.replace(tmp,path)

def load_pickle(path,default=None):
    try:
        with open(path,"rb") as f:
            return pickle.load(f)
    except Exception:
        return default

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

seed_all(cfg.seed)

CPU=os.cpu_count() or 1
N_EXACT=max(1,min(2,CPU))
TORCH_THREADS=max(1,min(8,CPU))

torch.set_num_threads(TORCH_THREADS)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True,warn_only=True)
except Exception:
    pass

print(
    f"CPU={CPU} | exact workers={N_EXACT} | "
    f"Torch threads={torch.get_num_threads()}"
)


# =====================================================================================
# 4. EXACT SIRS TEACHER
# =====================================================================================

@dataclass
class Rec:
    b:float
    g:float
    w:float
    N:int
    i0:int
    p:np.ndarray
    mt:float
    vt:float
    tv:bool


@lru_cache(None)
def topo(N):
    states=[
        (s,i)
        for i in range(1,N+1)
        for s in range(N-i+1)
    ]

    ix={x:j for j,x in enumerate(states)}
    M=len(states)

    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]

    db=np.zeros(M)
    dg=np.zeros(M)
    dw=np.zeros(M)
    qb=np.zeros(M)

    for j,(s,i) in enumerate(states):
        r=N-s-i

        if s:
            ir.append(j)
            ic.append(ix[(s-1,i+1)])
            rate=s*i/N
            ib.append(rate)
            db[j]=rate

        dg[j]=i

        if i==1:
            qb[j]=i
        else:
            rr.append(j)
            rc.append(ix[(s,i-1)])
            rb.append(i)

        if r:
            wr.append(j)
            wc.append(ix[(s+1,i)])
            wb.append(r)
            dw[j]=r

    A=lambda x,d=float:np.asarray(x,dtype=d)

    return (
        ix,M,
        A(ir,int),A(ic,int),A(ib),
        A(rr,int),A(rc,int),A(rb),
        A(wr,int),A(wc,int),A(wb),
        db,dg,dw,qb
    )


def refined(A,lu,b,transpose=False):
    b=np.asarray(b,dtype=np.float64)
    mode="T" if transpose else "N"

    x=lu.solve(b,trans=mode)

    for _ in range(cfg.refine):
        r=b-(A.T@x if transpose else A@x)

        if not np.all(np.isfinite(r)):
            break

        rel=np.linalg.norm(r,np.inf)/max(
            np.linalg.norm(b,np.inf),1.
        )

        if rel<1e-11:
            break

        x+=lu.solve(r,trans=mode)

    return np.asarray(x,dtype=np.float64)


def moment_factor(A,ordering):
    d=np.abs(A.diagonal())
    scale=1./np.maximum(d,np.finfo(float).tiny)

    As=(
        sparse.diags(scale)
        @A
    ).tocsc()

    return splu(
        As,
        permc_spec=ordering
    ),scale


def moment_solve(A,lu,scale,b):
    b=np.asarray(b,dtype=np.float64)

    x=lu.solve(
        scale*b
    )

    for _ in range(cfg.refine):
        r=b-A@x

        if not np.all(np.isfinite(r)):
            break

        rel=np.linalg.norm(r,np.inf)/max(
            np.linalg.norm(b,np.inf),1.
        )

        if rel<1e-11:
            break

        x+=lu.solve(
            scale*r
        )

    return np.asarray(x,dtype=np.float64)


def variance_system(T,q,A,lu,scale,m1):
    C=T.tocoo()
    off=C.row!=C.col

    source=np.bincount(
        C.row[off],
        weights=(
            C.data[off]
            *
            (
                m1[C.col[off]]
                -
                m1[C.row[off]]
            )**2
        ),
        minlength=T.shape[0]
    ).astype(np.float64)

    source+=q*m1*m1

    if (
        not np.all(np.isfinite(source))
        or source.min() < -1e-8
    ):
        raise ArithmeticError("Invalid variance-system RHS.")

    return moment_solve(
        A,lu,scale,
        np.maximum(source,0.)
    )


def exact_target(b,g,w,N,i0):
    (
        ix,M,ir,ic,ib,rr,rc,rb,
        wr,wc,wb,db,dg,dw,qb
    )=topo(N)

    rows=np.r_[ir,rr,wr,np.arange(M)]
    cols=np.r_[ic,rc,wc,np.arange(M)]

    vals=np.r_[
        b*ib,
        g*rb,
        w*wb,
        -(b*db+g*dg+w*dw)
    ]

    T=sparse.coo_matrix(
        (vals,(rows,cols)),
        shape=(M,M),
        dtype=np.float64
    ).tocsc()

    D1=sparse.coo_matrix(
        (b*ib,(ir,ic)),
        shape=(M,M),
        dtype=np.float64
    ).tocsc()

    D0=(T-D1).tocsc()
    q=g*qb
    initial=ix[(N-i0,i0)]

    # -------------------------------------------------------------------------
    # Truncated-with-overflow infection-count distribution
    # -------------------------------------------------------------------------

    A0=(-D0).tocsc()
    lu0=splu(
        A0,
        permc_spec="COLAMD"
    )

    bvec=refined(
        A0,lu0,q
    )

    v=np.zeros(M)
    v[initial]=1.

    D1T=D1.T.tocsr()
    p=np.zeros(N+2)

    for k in range(N+1):
        p[k]=v@bvec

        y=refined(
            A0,lu0,v,
            transpose=True
        )

        v=np.asarray(
            D1T@y
        ).ravel()

    p[-1]=v.sum()
    p[np.abs(p)<cfg.prob_tol]=0.

    if (
        not np.all(np.isfinite(p))
        or p.min() < -cfg.prob_tol
    ):
        raise RuntimeError("Invalid exact PMF.")

    p=np.maximum(p,0.)
    mass=p.sum()

    if (
        not np.isfinite(mass)
        or abs(mass-1.)>1e-5
    ):
        raise RuntimeError(
            f"Invalid exact mass={mass}."
        )

    p/=mass

    # -------------------------------------------------------------------------
    # Extinction-time moments
    # -------------------------------------------------------------------------

    A=(-T).tocsc()
    mt=vt=np.nan
    valid=False

    for ordering in ("COLAMD","MMD_AT_PLUS_A"):
        try:
            lu,scale=moment_factor(
                A,
                ordering
            )

            m1=moment_solve(
                A,lu,scale,
                np.ones(M)
            )

            mean=float(
                m1[initial]
            )

            if (
                not np.all(np.isfinite(m1))
                or mean<=0
            ):
                raise ArithmeticError("Invalid E(tau).")

            m2=moment_solve(
                A,lu,scale,
                2.*m1
            )

            second=float(
                m2[initial]
            )

            if (
                not np.all(np.isfinite(m2))
                or second<=0
            ):
                raise ArithmeticError("Invalid E(tau^2).")

            raw=(
                np.longdouble(second)
                -
                np.longdouble(mean)**2
            )

            tol=cfg.var_tol*max(
                abs(second),
                mean*mean,
                1.
            )

            if np.isfinite(raw) and raw>=-tol:
                var=max(float(raw),0.)
            else:
                vv=variance_system(
                    T,q,A,lu,scale,m1
                )
                var=float(vv[initial])

            if (
                not np.isfinite(var)
                or var<0
            ):
                raise ArithmeticError("Invalid Var(tau).")

            mt=mean
            vt=var
            valid=True
            break

        except Exception:
            pass

    return Rec(
        b,g,w,N,i0,p,mt,vt,valid
    )


# =====================================================================================
# 5. DESIGN + RESUMABLE EXACT TARGETS
# =====================================================================================

def design(n,Ns,seed):
    U=qmc.LatinHypercube(
        d=4,
        seed=seed
    ).random(n)

    scale=lambda x,a:a[0]+(a[1]-a[0])*x

    b=scale(U[:,0],cfg.beta)
    g=scale(U[:,1],cfg.gamma)
    w=scale(U[:,2],cfg.omega)
    f=scale(U[:,3],cfg.frac)

    Nv=np.tile(
        np.asarray(Ns),
        math.ceil(n/len(Ns))
    )[:n]

    rng=np.random.default_rng(seed+99)
    rng.shuffle(Nv)

    i0=np.asarray([
        int(
            np.clip(
                round(f[j]*Nv[j]),
                2,
                Nv[j]
            )
        )
        for j in range(n)
    ])

    for N in Ns:
        z=np.where(Nv==N)[0]

        if len(z):
            k=max(
                1,
                round(.25*len(z))
            )

            i0[
                rng.choice(
                    z,k,replace=False
                )
            ]=1

    return [
        (
            float(b[j]),
            float(g[j]),
            float(w[j]),
            int(Nv[j]),
            int(i0[j])
        )
        for j in range(n)
    ]


def records_match(records,configs,tol=1e-12):
    if records is None or len(records)!=len(configs):
        return False

    for r,x in zip(records,configs):
        b,g,w,N,i0=x

        if (
            abs(r.b-b)>tol
            or abs(r.g-g)>tol
            or abs(r.w-w)>tol
            or r.N!=N
            or r.i0!=i0
        ):
            return False

    return True


def exact_one(j,x):
    return j,exact_target(*x)


def exact_set(name,configs):
    full=EXACT/f"{name}_full.pkl"

    if full.exists():
        z=load_pickle(full)

        if records_match(z,configs):
            print(
                f"{name}: full cache loaded ({len(z):,})"
            )
            return z

    folder=EXACT/name
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    ans=[]

    for start in range(
        0,len(configs),EXACT_CHUNK
    ):
        end=min(
            start+EXACT_CHUNK,
            len(configs)
        )

        cc=configs[start:end]

        f=folder/f"chunk_{start:05d}_{end:05d}.pkl"

        part=(
            load_pickle(f)
            if f.exists()
            else None
        )

        if not records_match(part,cc):
            jobs=list(
                enumerate(cc)
            )

            # Expensive population sizes first.
            jobs.sort(
                key=lambda z:z[1][3],
                reverse=True
            )

            t0=time.perf_counter()

            result=Parallel(
                n_jobs=N_EXACT,
                backend="threading"
            )(
                delayed(exact_one)(j,x)
                for j,x in jobs
            )

            result.sort(
                key=lambda z:z[0]
            )

            part=[
                r for _,r in result
            ]

            atomic_pickle(
                part,f
            )

            print(
                f"{name} {start:5d}:{end:5d} | "
                f"{time.perf_counter()-t0:.1f}s | saved"
            )

        else:
            print(
                f"{name} {start:5d}:{end:5d} | cache"
            )

        ans.extend(part)

    if not records_match(ans,configs):
        raise RuntimeError(
            f"{name}: design/cache mismatch."
        )

    atomic_pickle(
        ans,full
    )

    return ans


# =====================================================================================
# 6. DATASETS
# =====================================================================================

train_cfg=design(
    cfg.n_train,
    cfg.train_N,
    cfg.seed+1
)

val_cfg=design(
    N_VAL,
    cfg.train_N,
    cfg.seed+2
)

seen_cfg=design(
    N_TEST_SEEN,
    cfg.train_N,
    cfg.seed+3
)

interp_cfg=design(
    N_TEST_INTERP,
    cfg.interp_N,
    cfg.seed+4
)

train=exact_set(
    "TRAIN_R10000",
    train_cfg
)

val=exact_set(
    "VALIDATION",
    val_cfg
)

test_seen=exact_set(
    "TEST_SEEN",
    seen_cfg
)

test_interp=exact_set(
    "TEST_INTERP",
    interp_cfg
)


def audit(records,name):
    bad=[
        r for r in records
        if (
            not np.all(np.isfinite(r.p))
            or abs(r.p.sum()-1.)>1e-6
        )
    ]

    if bad:
        raise RuntimeError(
            f"Invalid probability targets in {name}."
        )

    print(
        f"{name:14s} | n={len(records):5d} | "
        f"tau valid={sum(r.tv for r in records):5d}/{len(records):5d}"
    )


print("\nEXACT TARGET AUDIT")

for data,name in (
    (train,"TRAIN"),
    (val,"VALIDATION"),
    (test_seen,"TEST SEEN"),
    (test_interp,"TEST INTERP")
):
    audit(data,name)


# =====================================================================================
# 7. NETWORKS / PACKING
# =====================================================================================

def mlp(din,dout):
    L=[]
    d=din

    for _ in range(cfg.depth):
        L += [
            nn.Linear(d,cfg.width),
            nn.SiLU()
        ]
        d=cfg.width

    L.append(
        nn.Linear(d,dout)
    )

    return nn.Sequential(*L)


class HazardNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=mlp(6,1)

    def forward(self,x):
        return torch.sigmoid(
            self.net(x).squeeze(-1)
        )


class TauNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=mlp(5,2)

    def forward(self,x):
        return torch.nn.functional.softplus(
            self.net(x)
        )


def pack(records):
    groups={}

    for j,r in enumerate(records):
        groups.setdefault(r.N,[]).append(j)

    P={}

    for N,idx in groups.items():
        idx=np.asarray(idx,dtype=int)
        rr=[records[j] for j in idx]

        B=len(rr)
        K=N+1

        b=torch.tensor(
            [r.b for r in rr],
            dtype=torch.float32
        )[:,None]

        g=torch.tensor(
            [r.g for r in rr],
            dtype=torch.float32
        )[:,None]

        w=torch.tensor(
            [r.w for r in rr],
            dtype=torch.float32
        )[:,None]

        ns=torch.full(
            (B,1),
            N/N_SCALE,
            dtype=torch.float32
        )

        i0=torch.tensor(
            [r.i0/N for r in rr],
            dtype=torch.float32
        )[:,None]

        c=(
            torch.arange(
                K,
                dtype=torch.float32
            )/N
        )[None,:]

        Xh=torch.stack([
            b.expand(B,K),
            g.expand(B,K),
            w.expand(B,K),
            ns.expand(B,K),
            i0.expand(B,K),
            c.expand(B,K)
        ],dim=2).contiguous()

        Yp=torch.from_numpy(
            np.stack(
                [r.p for r in rr]
            ).astype(np.float32)
        )

        Xt=torch.column_stack([
            b[:,0],
            g[:,0],
            w[:,0],
            ns[:,0],
            i0[:,0]
        ]).contiguous()

        mask=torch.tensor(
            [r.tv for r in rr],
            dtype=torch.bool
        )

        Yt=np.zeros(
            (B,2),
            dtype=np.float32
        )

        for j,r in enumerate(rr):
            if r.tv:
                Yt[j]=np.log1p(
                    [r.mt,r.vt]
                )

        P[N]={
            "Xh":Xh,
            "Yp":Yp,
            "Xt":Xt,
            "Yt":torch.from_numpy(Yt),
            "mask":mask,
            "orig":idx,
            "n":B
        }

    return P


# =====================================================================================
# 8. PMF / TAIL / LOSS
# =====================================================================================

def reconstruct(h):
    B=h.shape[0]

    before=torch.cat([
        torch.ones(
            (B,1),
            dtype=h.dtype
        ),
        torch.cumprod(
            1-h[:,:-1],
            dim=1
        )
    ],dim=1)

    return torch.cat([
        before*h,
        torch.prod(
            1-h,
            dim=1,
            keepdim=True
        )
    ],dim=1)


def tail(p):
    return torch.flip(
        torch.cumsum(
            torch.flip(
                p[:,1:],
                dims=[1]
            ),
            dim=1
        ),
        dims=[1]
    )


def predict_p(hnet,X):
    B,K,_=X.shape

    h=hnet(
        X.reshape(B*K,6)
    ).reshape(B,K)

    return reconstruct(h)


def loss_batch(hnet,tnet,G,ii):
    X=G["Xh"][ii]
    Y=G["Yp"][ii]

    ph=predict_p(
        hnet,X
    )

    lp=torch.sum(
        (ph-Y)**2,
        dim=1
    ).mean()

    lr=torch.mean(
        (tail(ph)-tail(Y))**2,
        dim=1
    ).mean()

    mask=G["mask"][ii]

    if torch.any(mask):
        z=G["Yt"][ii][mask]

        zh=tnet(
            G["Xt"][ii][mask]
        )

        lt=torch.sum(
            (zh-z)**2
            /
            (1+z*z),
            dim=1
        ).mean()
    else:
        lt=torch.zeros(())

    return (
        lp
        +
        cfg.lambda_rho*lr
        +
        cfg.lambda_tau*lt
    )


def schedule(P,rng,shuffle=True):
    jobs=[]

    for N,G in P.items():
        idx=np.arange(
            G["n"]
        )

        if shuffle:
            rng.shuffle(idx)

        for s in range(
            0,len(idx),cfg.batch
        ):
            jobs.append(
                (N,idx[s:s+cfg.batch])
            )

    if shuffle:
        rng.shuffle(jobs)

    return jobs


@torch.no_grad()
def val_loss(h,t,V):
    h.eval()
    t.eval()

    total=0.
    n=0
    rng=np.random.default_rng(1)

    for N,ii in schedule(
        V,rng,False
    ):
        L=loss_batch(
            h,t,V[N],ii
        )

        total+=L.item()*len(ii)
        n+=len(ii)

    return total/n


TR=pack(train)
VA=pack(val)
TE=pack(test_seen)
TI=pack(test_interp)


# =====================================================================================
# 9. RESUMABLE NEURAL TRAINING
# =====================================================================================

def cpu_state(state):
    return {
        k:v.detach().cpu()
        for k,v in state.items()
    }


def fit(rep):
    name=f"rep{rep:02d}"

    final=MODELS/f"{name}_FINAL.pt"
    checkpoint=CKPT/f"{name}_CHECKPOINT.pt"

    h=HazardNet()
    t=TauNet()

    if final.exists():
        z=torch.load(
            final,
            map_location="cpu",
            weights_only=False
        )

        h.load_state_dict(
            z["hazard"]
        )

        t.load_state_dict(
            z["tau"]
        )

        print(
            f"{name}: FINAL model loaded | "
            f"best epoch={z['best_epoch']}"
        )

        return h,t,z

    seed=(
        cfg.seed
        +
        7001
        +
        rep*100003
    )

    seed_all(seed)

    h=HazardNet()
    t=TauNet()

    pars=(
        list(h.parameters())
        +
        list(t.parameters())
    )

    opt=torch.optim.AdamW(
        pars,
        lr=cfg.lr,
        weight_decay=cfg.wd
    )

    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        factor=.5,
        patience=15
    )

    rng=np.random.default_rng(
        seed+77
    )

    start_epoch=1
    best=np.inf
    best_epoch=0
    best_h=None
    best_t=None
    wait=0
    history=[]
    previous_elapsed=0.

    if checkpoint.exists():
        z=torch.load(
            checkpoint,
            map_location="cpu",
            weights_only=False
        )

        h.load_state_dict(
            z["h_current"]
        )
        t.load_state_dict(
            z["t_current"]
        )

        opt.load_state_dict(
            z["optimizer"]
        )
        sch.load_state_dict(
            z["scheduler"]
        )

        start_epoch=z["epoch"]+1

        best=z["best"]
        best_epoch=z["best_epoch"]
        best_h=z["best_h"]
        best_t=z["best_t"]
        wait=z["wait"]

        history=z["history"]
        previous_elapsed=z["elapsed"]

        rng.bit_generator.state=z["rng_state"]

        if "torch_rng" in z:
            torch.set_rng_state(
                z["torch_rng"]
            )

        print(
            f"{name}: RESUME epoch {start_epoch}"
        )

    start=time.perf_counter()
    stopped=cfg.epochs

    for epoch in range(
        start_epoch,
        cfg.epochs+1
    ):
        h.train()
        t.train()

        for N,ii in schedule(
            TR,rng,True
        ):
            opt.zero_grad(
                set_to_none=True
            )

            L=loss_batch(
                h,t,TR[N],ii
            )

            if not torch.isfinite(L):
                raise RuntimeError(
                    f"{name}: non-finite loss."
                )

            L.backward()

            torch.nn.utils.clip_grad_norm_(
                pars,
                cfg.clip
            )

            opt.step()

        v=val_loss(
            h,t,VA
        )

        sch.step(v)

        history.append(
            float(v)
        )

        if (
            best_h is None
            or v<best-cfg.delta
        ):
            best=float(v)
            best_epoch=epoch
            best_h=cpu_state(
                h.state_dict()
            )
            best_t=cpu_state(
                t.state_dict()
            )
            wait=0

        else:
            wait+=1

        if (
            epoch==1
            or epoch%CHECKPOINT_EVERY==0
        ):
            elapsed=(
                previous_elapsed
                +
                time.perf_counter()
                -
                start
            )

            atomic_torch({
                "epoch":epoch,
                "h_current":cpu_state(h.state_dict()),
                "t_current":cpu_state(t.state_dict()),
                "optimizer":opt.state_dict(),
                "scheduler":sch.state_dict(),
                "best":best,
                "best_epoch":best_epoch,
                "best_h":best_h,
                "best_t":best_t,
                "wait":wait,
                "history":history,
                "rng_state":rng.bit_generator.state,
                "torch_rng":torch.get_rng_state(),
                "elapsed":elapsed
            },checkpoint)

            print(
                f"{name} | ep={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait}"
            )

        if wait>=cfg.patience:
            stopped=epoch
            break

    elapsed=(
        previous_elapsed
        +
        time.perf_counter()
        -
        start
    )

    if best_h is None or best_t is None:
        raise RuntimeError(
            f"{name}: no valid best state."
        )

    h.load_state_dict(best_h)
    t.load_state_dict(best_t)

    payload={
        "hazard":best_h,
        "tau":best_t,
        "best_epoch":best_epoch,
        "stopped_epoch":stopped,
        "validation":best,
        "history":history,
        "training_sec":elapsed
    }

    atomic_torch(
        payload,
        final
    )

    if checkpoint.exists():
        checkpoint.unlink()

    print(
        f"{name}: COMPLETE | "
        f"best={best_epoch} | stop={stopped}"
    )

    return h,t,payload


# =====================================================================================
# 10. EVALUATION
# =====================================================================================

METRICS=(
    "E2",
    "E_rho",
    "overflow_error",
    "KL",
    "mean_tau_rel_error",
    "var_tau_rel_error"
)


@torch.no_grad()
def evaluate(h,t,P,records):
    h.eval()
    t.eval()

    n=len(records)

    out={
        k:np.full(
            n,
            np.nan,
            dtype=float
        )
        for k in METRICS
    }

    for N,G in P.items():
        for s in range(
            0,G["n"],cfg.batch
        ):
            e=min(
                s+cfg.batch,
                G["n"]
            )

            loc=np.arange(s,e)
            orig=G["orig"][loc]

            X=G["Xh"][loc]
            Y=G["Yp"][loc]

            Ph=predict_p(
                h,X
            )

            out["E2"][orig]=(
                torch.linalg.vector_norm(
                    Ph-Y,
                    dim=1
                ).numpy()
            )

            out["E_rho"][orig]=(
                torch.max(
                    torch.abs(
                        tail(Ph)-tail(Y)
                    ),
                    dim=1
                ).values.numpy()
            )

            out["overflow_error"][orig]=(
                torch.abs(
                    Ph[:,-1]-Y[:,-1]
                ).numpy()
            )

            Psafe=torch.clamp(
                Ph,
                min=1e-12,
                max=1.
            )

            KL=torch.where(
                Y>0,
                Y*torch.log(
                    Y/Psafe
                ),
                torch.zeros_like(Y)
            ).sum(dim=1)

            out["KL"][orig]=(
                KL.numpy()
            )

            mask=G["mask"][loc]

            if torch.any(mask):
                z=(
                    t(
                        G["Xt"][loc][mask]
                    )
                    .numpy()
                    .astype(np.float64)
                )

                moments=np.expm1(
                    np.clip(
                        z,
                        0.,
                        80.
                    )
                )

                valid_orig=orig[
                    mask.numpy()
                ]

                for q,j in enumerate(valid_orig):
                    r=records[int(j)]

                    out[
                        "mean_tau_rel_error"
                    ][j]=(
                        abs(
                            moments[q,0]-r.mt
                        )
                        /
                        r.mt
                    )

                    out[
                        "var_tau_rel_error"
                    ][j]=(
                        abs(
                            moments[q,1]-r.vt
                        )
                        /
                        max(
                            r.vt,
                            1e-300
                        )
                    )

    return out


# =====================================================================================
# 11. FIVE INDEPENDENT TRAINING REPLICATIONS
# =====================================================================================

PROGRESS_FILE=ROOT/"experiment_progress.pkl"

progress=load_pickle(
    PROGRESS_FILE,
    {"results":{}}
)

progress.setdefault(
    "results",
    {}
)

for rep in range(1,N_REP+1):
    key=f"rep{rep:02d}"

    if key in progress["results"]:
        print(
            f"{key}: complete — skipping"
        )
        continue

    print("\n"+"="*90)
    print(f"TRAINING REPLICATION {rep}/{N_REP}")
    print("="*90)

    h,t,meta=fit(rep)

    seen=evaluate(
        h,t,TE,test_seen
    )

    interp=evaluate(
        h,t,TI,test_interp
    )

    progress["results"][key]={
        "rep":rep,
        "seen":seen,
        "interpolation":interp,
        "training_sec":meta["training_sec"],
        "best_epoch":meta["best_epoch"],
        "stopped_epoch":meta["stopped_epoch"],
        "validation":meta["validation"]
    }

    atomic_pickle(
        progress,
        PROGRESS_FILE
    )

    print(
        "Seen | "
        f"E2={np.nanmedian(seen['E2']):.4g} | "
        f"E_rho={np.nanmedian(seen['E_rho']):.4g}"
    )

    print(
        "Interpolation | "
        f"E2={np.nanmedian(interp['E2']):.4g} | "
        f"E_rho={np.nanmedian(interp['E_rho']):.4g}"
    )

    del h,t


# =====================================================================================
# 12. ERROR MATRICES + HIERARCHICAL BOOTSTRAP
# =====================================================================================

def error_matrix(split,metric):
    return np.vstack([
        progress["results"][
            f"rep{rep:02d}"
        ][split][metric]
        for rep in range(1,N_REP+1)
    ])


def hier_ci(A,B=BOOT_B,seed=1):
    A=np.asarray(A,float)
    nr,nc=A.shape

    rng=np.random.default_rng(seed)
    boot=[]

    for _ in range(B):
        ir=rng.integers(
            0,nr,size=nr
        )

        ic=rng.integers(
            0,nc,size=nc
        )

        z=np.nanmedian(
            A[ir][:,ic]
        )

        if np.isfinite(z):
            boot.append(z)

    if not boot:
        return np.nan,np.nan

    return (
        float(np.quantile(boot,.025)),
        float(np.quantile(boot,.975))
    )


def summarize(A,seed):
    x=np.asarray(A,float)
    x=x[np.isfinite(x)]

    lo,hi=hier_ci(
        A,
        seed=seed
    )

    return {
        "n":len(x),
        "median":float(np.median(x)),
        "q1":float(np.quantile(x,.25)),
        "q3":float(np.quantile(x,.75)),
        "p95":float(np.quantile(x,.95)),
        "CI_low":lo,
        "CI_high":hi
    }


summary_rows=[]

for si,split in enumerate(
    ("seen","interpolation")
):
    for mi,metric in enumerate(METRICS):
        z=summarize(
            error_matrix(split,metric),
            cfg.seed+1000*si+mi
        )

        summary_rows.append({
            "split":split,
            "metric":metric,
            **z
        })


summary=pd.DataFrame(
    summary_rows
)

summary.to_csv(
    OUT/"full_numeric_summary.csv",
    index=False
)


def S(split,metric):
    return summary[
        (summary.split==split)
        &
        (summary.metric==metric)
    ].iloc[0]


# =====================================================================================
# 13. INTERPOLATION CONTRAST
#
# Difference:
#
#   median held-out-N error - median training-grid-N error
#
# Same training-replication resample is used in both terms.
# Configuration resampling is separate because the two test sets are independent.
# =====================================================================================

def transfer_contrast(
    seen,
    interp,
    B=BOOT_B,
    seed=1
):
    seen=np.asarray(seen,float)
    interp=np.asarray(interp,float)

    nr=seen.shape[0]

    if interp.shape[0]!=nr:
        raise ValueError("Replication counts differ.")

    rng=np.random.default_rng(seed)
    boot=[]

    for _ in range(B):
        ir=rng.integers(
            0,nr,size=nr
        )

        js=rng.integers(
            0,seen.shape[1],
            size=seen.shape[1]
        )

        ji=rng.integers(
            0,interp.shape[1],
            size=interp.shape[1]
        )

        d=(
            np.nanmedian(
                interp[ir][:,ji]
            )
            -
            np.nanmedian(
                seen[ir][:,js]
            )
        )

        if np.isfinite(d):
            boot.append(d)

    point=(
        np.nanmedian(interp)
        -
        np.nanmedian(seen)
    )

    return (
        float(point),
        float(np.quantile(boot,.025)),
        float(np.quantile(boot,.975))
    )


contrast=[]

for j,metric in enumerate(METRICS):
    est,lo,hi=transfer_contrast(
        error_matrix("seen",metric),
        error_matrix("interpolation",metric),
        seed=cfg.seed+5000+j
    )

    contrast.append({
        "metric":metric,
        "contrast":"interpolation minus seen",
        "estimate":est,
        "CI_low":lo,
        "CI_high":hi
    })


contrast=pd.DataFrame(
    contrast
)

contrast.to_csv(
    OUT/"interpolation_contrast.csv",
    index=False
)


# =====================================================================================
# 14. MAIN TABLE
# =====================================================================================

def fmt(z):
    return (
        f"{z['median']:.4g} "
        f"[{z['CI_low']:.4g}, {z['CI_high']:.4g}]"
    )


main_rows=[]

for split,label in (
    ("seen","Training-grid N"),
    ("interpolation","Held-out N")
):
    main_rows.append({
        "Test regime":label,

        "E2 median [95% CI]":
            fmt(S(split,"E2")),

        "E2 p95":
            f"{S(split,'E2')['p95']:.4g}",

        "Erho median [95% CI]":
            fmt(S(split,"E_rho")),

        "Erho p95":
            f"{S(split,'E_rho')['p95']:.4g}",

        "Rel E(tau)":
            fmt(S(
                split,
                "mean_tau_rel_error"
            )),

        "Rel Var(tau)":
            fmt(S(
                split,
                "var_tau_rel_error"
            ))
    })


main_table=pd.DataFrame(
    main_rows
)

main_table.to_csv(
    OUT/"main_table_5_1A.csv",
    index=False
)

(
    OUT/"main_table_5_1A.tex"
).write_text(
    main_table.to_latex(
        index=False,
        escape=False
    )
)

print("\n"+"="*115)
print("MAIN TABLE")
print("="*115)
print(main_table.to_string(index=False))


# =====================================================================================
# 15. RAW TEST ERRORS
# =====================================================================================

raw=[]

for rep in range(1,N_REP+1):
    key=f"rep{rep:02d}"

    for split,records in (
        ("seen",test_seen),
        ("interpolation",test_interp)
    ):
        z=progress["results"][
            key
        ][split]

        for j,r in enumerate(records):
            row={
                "rep":rep,
                "split":split,
                "index":j,
                "N":r.N,
                "i0":r.i0,
                "beta":r.b,
                "gamma":r.g,
                "omega":r.w,
                "R0":r.b/r.g,
                "tau_valid":r.tv
            }

            for metric in METRICS:
                row[metric]=z[metric][j]

            raw.append(row)


raw=pd.DataFrame(raw)

raw.to_csv(
    OUT/"raw_test_errors.csv",
    index=False
)


# =====================================================================================
# 16. ACCURACY BY POPULATION SIZE
# =====================================================================================

def ci_for_indices(
    split,
    metric,
    idx,
    seed
):
    A=error_matrix(
        split,
        metric
    )[:,idx]

    return summarize(
        A,
        seed
    )


byN=[]

for split,records in (
    ("seen",test_seen),
    ("interpolation",test_interp)
):
    Ns=sorted(
        {r.N for r in records}
    )

    for N in Ns:
        idx=np.asarray([
            j
            for j,r in enumerate(records)
            if r.N==N
        ],dtype=int)

        row={
            "split":split,
            "N":N,
            "n_config":len(idx)
        }

        for k,metric in enumerate(
            (
                "E2",
                "E_rho",
                "mean_tau_rel_error",
                "var_tau_rel_error"
            )
        ):
            z=ci_for_indices(
                split,
                metric,
                idx,
                cfg.seed+N+1000*k
            )

            row[f"{metric}_median"]=z["median"]
            row[f"{metric}_CI_low"]=z["CI_low"]
            row[f"{metric}_CI_high"]=z["CI_high"]

        byN.append(row)


byN=pd.DataFrame(
    byN
)

byN.to_csv(
    OUT/"accuracy_by_N.csv",
    index=False
)


# =====================================================================================
# 17. ROBUSTNESS BY EPIDEMIC REGIME
# =====================================================================================

raw["i0_group"]=np.where(
    raw.i0==1,
    "i0=1",
    "i0>1"
)

raw["R0_regime"]=pd.cut(
    raw.R0,
    [-np.inf,1.,2.,np.inf],
    labels=[
        "R0<1",
        "1<=R0<2",
        "R0>=2"
    ]
)


rob=[]

for stype,column in (
    ("initial condition","i0_group"),
    ("epidemic regime","R0_regime")
):
    for (
        split,
        level
    ),z in raw.groupby(
        ["split",column],
        observed=True
    ):
        rob.append({
            "stratum_type":stype,
            "stratum":str(level),
            "split":split,
            "n":len(z),
            "median_E2":float(
                np.nanmedian(z.E2)
            ),
            "p95_E2":float(
                np.nanquantile(z.E2,.95)
            ),
            "median_Erho":float(
                np.nanmedian(z.E_rho)
            ),
            "p95_Erho":float(
                np.nanquantile(z.E_rho,.95)
            )
        })


pd.DataFrame(rob).to_csv(
    OUT/"robustness_by_stratum.csv",
    index=False
)


# =====================================================================================
# 18. TRAINING DIAGNOSTICS
# =====================================================================================

training=[]

for rep in range(1,N_REP+1):
    z=progress["results"][
        f"rep{rep:02d}"
    ]

    training.append({
        "rep":rep,
        "training_sec":z["training_sec"],
        "best_epoch":z["best_epoch"],
        "stopped_epoch":z["stopped_epoch"],
        "validation":z["validation"]
    })


pd.DataFrame(training).to_csv(
    OUT/"training_diagnostics.csv",
    index=False
)


# =====================================================================================
# 19. EXTINCTION-TIME RESOLUTION
# =====================================================================================

tau_resolution=pd.DataFrame([
    {
        "dataset":"train",
        "valid":sum(r.tv for r in train),
        "total":len(train)
    },
    {
        "dataset":"validation",
        "valid":sum(r.tv for r in val),
        "total":len(val)
    },
    {
        "dataset":"seen test",
        "valid":sum(r.tv for r in test_seen),
        "total":len(test_seen)
    },
    {
        "dataset":"interpolation test",
        "valid":sum(r.tv for r in test_interp),
        "total":len(test_interp)
    }
])

tau_resolution[
    "fraction"
]=(
    tau_resolution.valid
    /
    tau_resolution.total
)

tau_resolution.to_csv(
    OUT/"tau_resolution.csv",
    index=False
)


# =====================================================================================
# 20. MAIN 2x2 FIGURE
# =====================================================================================

plt.rcParams.update({
    "font.size":10.5,
    "axes.spines.top":False,
    "axes.spines.right":False
})

fig,axs=plt.subplots(
    2,2,
    figsize=(11.5,8.0)
)

spec=[
    (
        "E2",
        r"$E_2$",
        "(A) Distributional error"
    ),
    (
        "E_rho",
        r"$E_\rho$",
        "(B) Tail-risk error"
    ),
    (
        "mean_tau_rel_error",
        "Relative error",
        r"(C) $E(\tau)$"
    ),
    (
        "var_tau_rel_error",
        "Relative error",
        r"(D) $\mathrm{Var}(\tau)$"
    )
]


for ax,(metric,ylabel,title) in zip(
    axs.flat,
    spec
):
    for split,label,color,marker in (
        (
            "seen",
            "Training-grid $N$",
            "#555555",
            "o"
        ),
        (
            "interpolation",
            "Held-out $N$",
            "#0072B2",
            "D"
        )
    ):
        z=byN[
            byN.split==split
        ].sort_values("N")

        x=z.N.to_numpy()
        med=z[
            f"{metric}_median"
        ].to_numpy()

        lo=z[
            f"{metric}_CI_low"
        ].to_numpy()

        hi=z[
            f"{metric}_CI_high"
        ].to_numpy()

        ok=(
            np.isfinite(med)
            &
            np.isfinite(lo)
            &
            np.isfinite(hi)
        )

        ax.errorbar(
            x[ok],
            np.maximum(
                med[ok],
                1e-12
            ),
            yerr=np.vstack([
                np.maximum(
                    med[ok]-lo[ok],
                    0
                ),
                np.maximum(
                    hi[ok]-med[ok],
                    0
                )
            ]),
            fmt=marker+"-",
            color=color,
            lw=1.8,
            markersize=4,
            capsize=2,
            label=label
        )

    ax.set_yscale("log")
    ax.set_xlabel(
        "Population size $N$"
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=.15)
    ax.legend(
        frameon=False,
        fontsize=9
    )


fig.suptitle(
    r"Exact-Teacher Accuracy and Population-Size Generalization "
    r"at $R=10{,}000$",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    OUT/"figure_5_1A_main.pdf",
    bbox_inches="tight"
)

plt.savefig(
    OUT/"figure_5_1A_main.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 21. CAPTION + FINAL RESULT OBJECT
# =====================================================================================

caption=r"""
Accuracy of the exact-teacher neural emulator at R=10,000. Training-grid
population sizes are N=40,60,...,400, whereas the held-out population sizes
N=50,70,...,390 are never used for teacher construction and therefore provide
a direct interpolation assessment. Panels report distributional, tail-risk,
and extinction-time errors as functions of population size. Points are
medians and error bars are hierarchical bootstrap 95% confidence intervals
over independent neural-training replications and exact test configurations.
Extinction-time errors are reported only for configurations for which the
exact sparse linear-system calculation returned numerically admissible moment
estimates.
""".strip()

(
    OUT/"figure_5_1A_caption.txt"
).write_text(caption)


atomic_pickle(
    {
        "scientific_config":SCIENTIFIC_CONFIG,
        "signature":SIG,
        "summary":summary,
        "main_table":main_table,
        "interpolation_contrast":contrast,
        "accuracy_by_N":byN,
        "tau_resolution":tau_resolution
    },
    OUT/"section_5_1A_final_results.pkl"
)


# =====================================================================================
# 22. FINAL STATUS
# =====================================================================================

print("\n"+"="*105)
print("5.1-A COMPLETE")
print("="*105)

print("R:",f"{cfg.n_train:,}")
print("Training N:",cfg.train_N)
print("Held-out N:",cfg.interp_N)
print("Replications:",N_REP)

print(
    "Training configurations:",
    len(train)
)

print(
    "Validation configurations:",
    len(val)
)

print(
    "Seen-N test configurations:",
    len(test_seen)
)

print(
    "Held-out-N test configurations:",
    len(test_interp)
)

print("\nMain table:")
print(OUT/"main_table_5_1A.tex")

print("\nMain figure:")
print(OUT/"figure_5_1A_main.pdf")

print("\nAdditional diagnostics:")
print(OUT/"full_numeric_summary.csv")
print(OUT/"raw_test_errors.csv")
print(OUT/"accuracy_by_N.csv")
print(OUT/"interpolation_contrast.csv")
print(OUT/"robustness_by_stratum.csv")
print(OUT/"training_diagnostics.csv")
print(OUT/"tau_resolution.csv")

print(
    "\nDisconnect -> rerun cell -> choose 1 = RESUME.\n"
    "Scientific implementation changed -> increment CODE_VERSION -> "
    "choose 2 = START NEW."
)

print("="*105)

Mounted at /content/drive

1 = RESUME latest run
2 = START NEW run from zero
Choose 1 or 2: 2
ROOT: /content/drive/MyDrive/StatisticalLearning/Experiment_5_1A_JASA/run_20260823_085220
Signature: 1b35c2859a536121
CPU=2 | exact workers=2 | Torch threads=2


In [ ]:
# =====================================================================================
# FAILURE ANALYSIS — WORST 5% AND 10%
#
# Primary failure metric: E_rho
#
# Includes R0 = beta/gamma:
#   - in tables
#   - in regime diagnostics
#   - in worst-case PMF titles
#   - in worst-case tail titles
# =====================================================================================

from pathlib import Path
import pickle, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RESULT_DIR=Path(
    "results_section_5_1A_R5000_tailaware_R0"
)

RESULT_FILE=(
    RESULT_DIR
    /
    "section_5_1A_results.pkl"
)

FAIL_METRIC="E_rho"
FRACTIONS=(.05,.10)


with open(
    RESULT_FILE,
    "rb"
) as f:

    D=pickle.load(f)


ALL=D["all_test"]
INTERP=D["interpolation"]


# =====================================================================================
# 1. DATA FRAMES
# =====================================================================================

def to_df(results):

    keep=[
        "index",
        "split",

        "N",
        "i0",
        "i0_fraction",

        "beta",
        "gamma",
        "omega",
        "R0",

        "E2",
        "E_rho",
        "E_overflow",
        "KL",

        "mean_tau_relative_error",
        "var_tau_relative_error",

        "exact_overflow",
        "entropy",
        "bimodality",
        "tau_valid"
    ]

    return pd.DataFrame([
        {
            k:r.get(k,np.nan)
            for k in keep
        }
        for r in results
    ])


df_all=to_df(ALL)
df_interp=to_df(INTERP)


# =====================================================================================
# 2. WORST FRACTIONS
# =====================================================================================

def worst_fraction(
    df,
    fraction,
    metric=FAIL_METRIC
):

    n=max(
        1,
        int(
            math.ceil(
                fraction*len(df)
            )
        )
    )

    return (
        df
        .sort_values(
            metric,
            ascending=False
        )
        .head(n)
        .copy()
    )


W={}


for scope,df in [
    ("all",df_all),
    ("interpolation",df_interp)
]:

    print("\n"+"="*115)
    print(scope.upper())
    print("="*115)

    for frac in FRACTIONS:

        w=worst_fraction(
            df,
            frac
        )

        W[(scope,frac)]=w

        print(
            f"\nWorst {int(frac*100)}% by {FAIL_METRIC}: "
            f"{len(w)}/{len(df)}"
        )

        print(
            f"E_rho: "
            f"median={w.E_rho.median():.4f}, "
            f"min={w.E_rho.min():.4f}, "
            f"max={w.E_rho.max():.4f}"
        )

        print(
            f"R0: "
            f"median={w.R0.median():.3f}, "
            f"IQR=({w.R0.quantile(.25):.3f},"
            f"{w.R0.quantile(.75):.3f}), "
            f"range=({w.R0.min():.3f},"
            f"{w.R0.max():.3f})"
        )

        print(
            "N counts:",
            w["N"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        print(
            f"i0=1 fraction: "
            f"{(w.i0==1).mean():.3f}"
        )

        print(
            f"median beta={w.beta.median():.3f}, "
            f"gamma={w.gamma.median():.3f}, "
            f"omega={w.omega.median():.3f}"
        )

        print(
            f"median overflow="
            f"{w.exact_overflow.median():.4f}, "
            f"entropy="
            f"{w.entropy.median():.3f}, "
            f"bimodality="
            f"{w.bimodality.median():.3f}"
        )

        w.to_csv(
            RESULT_DIR
            /
            f"worst_{int(frac*100)}pct_{scope}.csv",
            index=False
        )


# =====================================================================================
# 3. ALL TEST VS WORST 10%
# =====================================================================================

w10=W[("all",.10)]

compare_cols=[
    "N",
    "i0_fraction",

    "beta",
    "gamma",
    "omega",
    "R0",

    "exact_overflow",
    "entropy",
    "bimodality",

    "E2",
    "E_rho",
    "KL"
]


comparison=pd.DataFrame({

    "all_test_median":
        df_all[
            compare_cols
        ].median(),

    "worst10_median":
        w10[
            compare_cols
        ].median(),

    "worst10_Q25":
        w10[
            compare_cols
        ].quantile(.25),

    "worst10_Q75":
        w10[
            compare_cols
        ].quantile(.75)

})


print("\n"+"="*115)
print("ALL TEST VERSUS WORST 10%")
print("="*115)

display(comparison)


comparison.to_csv(
    RESULT_DIR
    /
    "failure_regime_summary.csv"
)


# =====================================================================================
# 4. FAILURE DIAGNOSTICS
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


# ------------------------------------------------------------------
# A. Distribution of E_rho
# ------------------------------------------------------------------

axs[0,0].hist(
    df_all.E_rho,
    bins=35,
    color=".72",
    edgecolor="white"
)

q90=df_all.E_rho.quantile(.90)
q95=df_all.E_rho.quantile(.95)

axs[0,0].axvline(
    q90,
    color="#E69F00",
    ls="--",
    label="90th percentile"
)

axs[0,0].axvline(
    q95,
    color="#D55E00",
    ls="--",
    label="95th percentile"
)

axs[0,0].set_xlabel(
    r"$E_\rho$"
)

axs[0,0].set_ylabel(
    "Number of configurations"
)

axs[0,0].set_title(
    "(A) Tail-error distribution"
)

axs[0,0].legend(
    frameon=False
)


# ------------------------------------------------------------------
# B. Failure concentration by N
# ------------------------------------------------------------------

allN=df_all.groupby(
    "N"
).size()

badN=w10.groupby(
    "N"
).size()

rate=(
    badN.reindex(
        allN.index,
        fill_value=0
    )
    /
    allN
)

axs[0,1].bar(
    rate.index,
    rate.values,
    width=14,
    color="#0072B2"
)

axs[0,1].set_xlabel(
    "Population size $N$"
)

axs[0,1].set_ylabel(
    "Fraction in worst 10%"
)

axs[0,1].set_title(
    "(B) Failure concentration by $N$"
)


# ------------------------------------------------------------------
# C. R0 versus omega
# ------------------------------------------------------------------

axs[0,2].scatter(
    df_all.R0,
    df_all.omega,
    s=15,
    alpha=.18,
    color=".5",
    label="All test"
)

axs[0,2].scatter(
    w10.R0,
    w10.omega,
    s=32,
    alpha=.85,
    color="#D55E00",
    label="Worst 10%"
)

axs[0,2].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[0,2].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[0,2].set_ylabel(
    r"$\omega$"
)

axs[0,2].set_title(
    r"(C) Failure regime in $(R_0,\omega)$"
)

axs[0,2].legend(
    frameon=False
)


# ------------------------------------------------------------------
# D. R0 versus E_rho
# ------------------------------------------------------------------

axs[1,0].scatter(
    df_all.R0,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,0].scatter(
    w10.R0,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,0].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[1,0].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[1,0].set_ylabel(
    r"$E_\rho$"
)

axs[1,0].set_title(
    r"(D) Tail error versus $R_0$"
)


# ------------------------------------------------------------------
# E. Exact overflow versus E_rho
# ------------------------------------------------------------------

axs[1,1].scatter(
    df_all.exact_overflow,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,1].scatter(
    w10.exact_overflow,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,1].set_xlabel(
    r"Exact $P(C>N)$"
)

axs[1,1].set_ylabel(
    r"$E_\rho$"
)

axs[1,1].set_title(
    "(E) Overflow risk"
)


# ------------------------------------------------------------------
# F. Bimodality versus E_rho
# ------------------------------------------------------------------

axs[1,2].scatter(
    df_all.bimodality,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,2].scatter(
    w10.bimodality,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,2].set_xlabel(
    "Bimodality score"
)

axs[1,2].set_ylabel(
    r"$E_\rho$"
)

axs[1,2].set_title(
    "(F) Distributional shape"
)


plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_failure_diagnostics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 5. SIX WORST CASES
# =====================================================================================

worst6=sorted(
    ALL,
    key=lambda r:r[FAIL_METRIC],
    reverse=True
)[:6]


print("\n"+"="*115)
print("SIX WORST TEST CONFIGURATIONS")
print("="*115)


for j,r in enumerate(
    worst6,1
):

    print(
        f"{j}. "
        f"{r['split']:13s} | "
        f"N={r['N']:3d}, "
        f"i0={r['i0']:3d} | "
        f"beta={r['beta']:.3f}, "
        f"gamma={r['gamma']:.3f}, "
        f"omega={r['omega']:.3f} | "
        f"R0={r['R0']:.3f} | "
        f"E2={r['E2']:.4f}, "
        f"E_rho={r['E_rho']:.4f}, "
        f"overflow={r['exact_overflow']:.4f}"
    )


# =====================================================================================
# 6. WORST-6 PMFs — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.bar(
        c,
        r["exact_p"][:-1],
        color=".82",
        width=.85,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_p"][:-1],
        color="#D55E00",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_2={r['E2']:.3f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Infection count $c$"
    )

    ax.set_ylabel(
        "Probability mass"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_pmfs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 7. WORST-6 TAILS — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.plot(
        c,
        r["exact_tail"],
        color="black",
        lw=2,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_tail"],
        "--",
        color="#0072B2",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_ylim(
        -.01,1.01
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Threshold $c$"
    )

    ax.set_ylabel(
        r"$P(C>c)$"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_tails.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nFailure analysis saved to:")
print(RESULT_DIR.resolve())

In [ ]:
# =====================================================================================
# FAILURE ANALYSIS — WORST 5% AND 10%
#
# Primary failure metric: E_rho
#
# Includes R0 = beta/gamma:
#   - in tables
#   - in regime diagnostics
#   - in worst-case PMF titles
#   - in worst-case tail titles
# =====================================================================================

from pathlib import Path
import pickle, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RESULT_DIR=Path(
    "results_section_5_1A_R5000_tailaware_R0"
)

RESULT_FILE=(
    RESULT_DIR
    /
    "section_5_1A_results.pkl"
)

FAIL_METRIC="E_rho"
FRACTIONS=(.05,.10)


with open(
    RESULT_FILE,
    "rb"
) as f:

    D=pickle.load(f)


ALL=D["all_test"]
INTERP=D["interpolation"]


# =====================================================================================
# 1. DATA FRAMES
# =====================================================================================

def to_df(results):

    keep=[
        "index",
        "split",

        "N",
        "i0",
        "i0_fraction",

        "beta",
        "gamma",
        "omega",
        "R0",

        "E2",
        "E_rho",
        "E_overflow",
        "KL",

        "mean_tau_relative_error",
        "var_tau_relative_error",

        "exact_overflow",
        "entropy",
        "bimodality",
        "tau_valid"
    ]

    return pd.DataFrame([
        {
            k:r.get(k,np.nan)
            for k in keep
        }
        for r in results
    ])


df_all=to_df(ALL)
df_interp=to_df(INTERP)


# =====================================================================================
# 2. WORST FRACTIONS
# =====================================================================================

def worst_fraction(
    df,
    fraction,
    metric=FAIL_METRIC
):

    n=max(
        1,
        int(
            math.ceil(
                fraction*len(df)
            )
        )
    )

    return (
        df
        .sort_values(
            metric,
            ascending=False
        )
        .head(n)
        .copy()
    )


W={}


for scope,df in [
    ("all",df_all),
    ("interpolation",df_interp)
]:

    print("\n"+"="*115)
    print(scope.upper())
    print("="*115)

    for frac in FRACTIONS:

        w=worst_fraction(
            df,
            frac
        )

        W[(scope,frac)]=w

        print(
            f"\nWorst {int(frac*100)}% by {FAIL_METRIC}: "
            f"{len(w)}/{len(df)}"
        )

        print(
            f"E_rho: "
            f"median={w.E_rho.median():.4f}, "
            f"min={w.E_rho.min():.4f}, "
            f"max={w.E_rho.max():.4f}"
        )

        print(
            f"R0: "
            f"median={w.R0.median():.3f}, "
            f"IQR=({w.R0.quantile(.25):.3f},"
            f"{w.R0.quantile(.75):.3f}), "
            f"range=({w.R0.min():.3f},"
            f"{w.R0.max():.3f})"
        )

        print(
            "N counts:",
            w["N"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        print(
            f"i0=1 fraction: "
            f"{(w.i0==1).mean():.3f}"
        )

        print(
            f"median beta={w.beta.median():.3f}, "
            f"gamma={w.gamma.median():.3f}, "
            f"omega={w.omega.median():.3f}"
        )

        print(
            f"median overflow="
            f"{w.exact_overflow.median():.4f}, "
            f"entropy="
            f"{w.entropy.median():.3f}, "
            f"bimodality="
            f"{w.bimodality.median():.3f}"
        )

        w.to_csv(
            RESULT_DIR
            /
            f"worst_{int(frac*100)}pct_{scope}.csv",
            index=False
        )


# =====================================================================================
# 3. ALL TEST VS WORST 10%
# =====================================================================================

w10=W[("all",.10)]

compare_cols=[
    "N",
    "i0_fraction",

    "beta",
    "gamma",
    "omega",
    "R0",

    "exact_overflow",
    "entropy",
    "bimodality",

    "E2",
    "E_rho",
    "KL"
]


comparison=pd.DataFrame({

    "all_test_median":
        df_all[
            compare_cols
        ].median(),

    "worst10_median":
        w10[
            compare_cols
        ].median(),

    "worst10_Q25":
        w10[
            compare_cols
        ].quantile(.25),

    "worst10_Q75":
        w10[
            compare_cols
        ].quantile(.75)

})


print("\n"+"="*115)
print("ALL TEST VERSUS WORST 10%")
print("="*115)

display(comparison)


comparison.to_csv(
    RESULT_DIR
    /
    "failure_regime_summary.csv"
)


# =====================================================================================
# 4. FAILURE DIAGNOSTICS
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


# ------------------------------------------------------------------
# A. Distribution of E_rho
# ------------------------------------------------------------------

axs[0,0].hist(
    df_all.E_rho,
    bins=35,
    color=".72",
    edgecolor="white"
)

q90=df_all.E_rho.quantile(.90)
q95=df_all.E_rho.quantile(.95)

axs[0,0].axvline(
    q90,
    color="#E69F00",
    ls="--",
    label="90th percentile"
)

axs[0,0].axvline(
    q95,
    color="#D55E00",
    ls="--",
    label="95th percentile"
)

axs[0,0].set_xlabel(
    r"$E_\rho$"
)

axs[0,0].set_ylabel(
    "Number of configurations"
)

axs[0,0].set_title(
    "(A) Tail-error distribution"
)

axs[0,0].legend(
    frameon=False
)


# ------------------------------------------------------------------
# B. Failure concentration by N
# ------------------------------------------------------------------

allN=df_all.groupby(
    "N"
).size()

badN=w10.groupby(
    "N"
).size()

rate=(
    badN.reindex(
        allN.index,
        fill_value=0
    )
    /
    allN
)

axs[0,1].bar(
    rate.index,
    rate.values,
    width=14,
    color="#0072B2"
)

axs[0,1].set_xlabel(
    "Population size $N$"
)

axs[0,1].set_ylabel(
    "Fraction in worst 10%"
)

axs[0,1].set_title(
    "(B) Failure concentration by $N$"
)


# ------------------------------------------------------------------
# C. R0 versus omega
# ------------------------------------------------------------------

axs[0,2].scatter(
    df_all.R0,
    df_all.omega,
    s=15,
    alpha=.18,
    color=".5",
    label="All test"
)

axs[0,2].scatter(
    w10.R0,
    w10.omega,
    s=32,
    alpha=.85,
    color="#D55E00",
    label="Worst 10%"
)

axs[0,2].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[0,2].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[0,2].set_ylabel(
    r"$\omega$"
)

axs[0,2].set_title(
    r"(C) Failure regime in $(R_0,\omega)$"
)

axs[0,2].legend(
    frameon=False
)


# ------------------------------------------------------------------
# D. R0 versus E_rho
# ------------------------------------------------------------------

axs[1,0].scatter(
    df_all.R0,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,0].scatter(
    w10.R0,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,0].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[1,0].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[1,0].set_ylabel(
    r"$E_\rho$"
)

axs[1,0].set_title(
    r"(D) Tail error versus $R_0$"
)


# ------------------------------------------------------------------
# E. Exact overflow versus E_rho
# ------------------------------------------------------------------

axs[1,1].scatter(
    df_all.exact_overflow,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,1].scatter(
    w10.exact_overflow,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,1].set_xlabel(
    r"Exact $P(C>N)$"
)

axs[1,1].set_ylabel(
    r"$E_\rho$"
)

axs[1,1].set_title(
    "(E) Overflow risk"
)


# ------------------------------------------------------------------
# F. Bimodality versus E_rho
# ------------------------------------------------------------------

axs[1,2].scatter(
    df_all.bimodality,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,2].scatter(
    w10.bimodality,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,2].set_xlabel(
    "Bimodality score"
)

axs[1,2].set_ylabel(
    r"$E_\rho$"
)

axs[1,2].set_title(
    "(F) Distributional shape"
)


plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_failure_diagnostics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 5. SIX WORST CASES
# =====================================================================================

worst6=sorted(
    ALL,
    key=lambda r:r[FAIL_METRIC],
    reverse=True
)[:6]


print("\n"+"="*115)
print("SIX WORST TEST CONFIGURATIONS")
print("="*115)


for j,r in enumerate(
    worst6,1
):

    print(
        f"{j}. "
        f"{r['split']:13s} | "
        f"N={r['N']:3d}, "
        f"i0={r['i0']:3d} | "
        f"beta={r['beta']:.3f}, "
        f"gamma={r['gamma']:.3f}, "
        f"omega={r['omega']:.3f} | "
        f"R0={r['R0']:.3f} | "
        f"E2={r['E2']:.4f}, "
        f"E_rho={r['E_rho']:.4f}, "
        f"overflow={r['exact_overflow']:.4f}"
    )


# =====================================================================================
# 6. WORST-6 PMFs — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.bar(
        c,
        r["exact_p"][:-1],
        color=".82",
        width=.85,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_p"][:-1],
        color="#D55E00",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_2={r['E2']:.3f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Infection count $c$"
    )

    ax.set_ylabel(
        "Probability mass"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_pmfs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 7. WORST-6 TAILS — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.plot(
        c,
        r["exact_tail"],
        color="black",
        lw=2,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_tail"],
        "--",
        color="#0072B2",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_ylim(
        -.01,1.01
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Threshold $c$"
    )

    ax.set_ylabel(
        r"$P(C>c)$"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_tails.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nFailure analysis saved to:")
print(RESULT_DIR.resolve())